In [ ]:
from pathlib import Path
import numpy as np
from numpy import linalg as LA
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, IntSlider
%matplotlib widget

In [ ]:
def noisy_identity_transition_matrix(n, sigma=0.05):
    P = np.eye(n) + sigma * np.random.randn(n, n)
    P = np.abs(P)                     # no negatives
    P = P / P.sum(axis=1, keepdims=True)
    return P


def structured_transition_matrix(n, decay=0.5, noise=0.1):
    """
    Create an n x n transition matrix where:
    - diagonal values are strongest
    - off-diagonals decay with distance from the diagonal
    - randomness is controlled by `noise`

    decay: float in (0, 1). Lower = sharper drop-off away from diagonal.
    noise: magnitude of random noise added to break perfect structure.
    """

    # distance-based deterministic kernel
    i = np.arange(n)
    dist = np.abs(i[:, None] - i[None, :])
    K = decay ** dist   # (decay^0 on diagonal, decay^1 next, etc.)
    
    if noise > 0:
        # add some random variation
        K = K + noise * np.random.rand(n, n)

        # ensure positivity
        K = np.clip(K, 1e-12, None)

    # normalize rows → row stochastic
    P = K / K.sum(axis=1, keepdims=True)

    return P


def gridworld_transition_matrix(rows, cols, stay_prob=0.1, stochasticity=0.0):
    """
    Create a 2D gridworld transition matrix P of shape (rows*cols, rows*cols)
    
    Arguments:
        rows, cols: size of the grid
        stay_prob: probability to stay in the same cell
        stochasticity: probability of random transitions to neighboring cells (0 = deterministic)
    
    Returns:
        P: transition matrix, row-stochastic
    """
    n_states = rows * cols
    P = np.zeros((n_states, n_states))

    # Possible actions: up, down, left, right, up_right, up_left, down_right, down_left
    actions = [(-1, 0), (1, 0), (0, -1), (0, 1)]
    
    for r in range(rows):
        for c in range(cols):
            s = r * cols + c
            neighbors = []

            for dr, dc in actions:
                nr, nc = r + dr, c + dc
                # Check grid bounds
                if 0 <= nr < rows and 0 <= nc < cols:
                    neighbors.append(nr * cols + nc)
            
            # Assign probabilities
            n_neighbors = len(neighbors)
            for ns in neighbors:
                P[s, ns] = (1 - stay_prob) * (1 - stochasticity)  / n_neighbors
            
            # Random stochastic component
            if stochasticity > 0:
                for ns in neighbors:
                    P[s, ns] += stochasticity / n_neighbors
            
            # Probability to stay in place
            P[s, s] = stay_prob
    
    # Ensure rows sum to 1
    P = P / P.sum(axis=1, keepdims=True)
    return P

def make_four_rooms(rows, cols, wall_thickness=1, door_size=1, border=1):
    free = np.ones((rows, cols), dtype=bool)

    # outer border
    if border > 0:
        free[:border, :] = False
        free[-border:, :] = False
        free[:, :border] = False
        free[:, -border:] = False

    r0, r1 = border, rows - border
    c0, c1 = border, cols - border

    r_mid = (r0 + r1 - 1) // 2
    c_mid = (c0 + c1 - 1) // 2

    # cross walls
    free[r_mid:r_mid + wall_thickness, c0:c1] = False
    free[r0:r1, c_mid:c_mid + wall_thickness] = False

    def clamp(x, lo, hi):
        return int(max(lo, min(hi, x)))

    span_r = r1 - r0
    span_c = c1 - c0

    door_c1 = clamp(c0 + span_c // 3,     c0, c1 - door_size)
    door_c2 = clamp(c0 + 2 * span_c // 3, c0, c1 - door_size)
    door_r1 = clamp(r0 + span_r // 3,     r0, r1 - door_size)
    door_r2 = clamp(r0 + 2 * span_r // 3, r0, r1 - door_size)

    free[r_mid:r_mid + wall_thickness, door_c1:door_c1 + door_size] = True
    free[r_mid:r_mid + wall_thickness, door_c2:door_c2 + door_size] = True
    free[door_r1:door_r1 + door_size, c_mid:c_mid + wall_thickness] = True
    free[door_r2:door_r2 + door_size, c_mid:c_mid + wall_thickness] = True

    return free

def gridworld_transition_matrix_with_walls(rows, cols, free, stay_prob=0.0, remove_wall_states=True):
    N = rows * cols
    P = np.zeros((N, N))

    def idx(r, c): return r * cols + c
    actions = [(-1,0),(1,0),(0,-1),(0,1)]

    for r in range(rows):
        for c in range(cols):
            i = idx(r, c)

            if not free[r, c]:
                P[i, i] = 0.0
                continue

            nbrs = []
            for dr, dc in actions:
                rr, cc = r + dr, c + dc
                if 0 <= rr < rows and 0 <= cc < cols and free[rr, cc]:
                    nbrs.append(idx(rr, cc))

            if not nbrs:
                P[i, i] = 0.0
                continue

            P[i, nbrs] = (1.0 - stay_prob) / len(nbrs)
            P[i, i] += stay_prob

    # make sure rows sum to 1
    row_sums = P.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1.0  # avoid division by zero
    P /= row_sums

    # Optionally remove wall states from the transition matrix
    if remove_wall_states:
        wall_indices = [idx(r, c) for r in range(rows) for c in range(cols) if not free[r, c]]
        P = np.delete(P, wall_indices, axis=0)
        P = np.delete(P, wall_indices, axis=1)

    return P

def setup(n:int, mixing_coef:float=0.5, noise_coef:float=0.1, remove_wall_states:bool=True):
    """
    Set up the environment transition matrix and successor representation.
    Args:
        n: number of states
        gamma: discount factor
        trans_decay: decay factor for transition matrix structure. Lower = more local transitions (diagonal).
        trans_noise: noise level for transition matrix. Higher = more random transitions.
    Returns:
        P: transition matrix (n x n)
    """
    # P = structured_transition_matrix(n, decay=trans_decay, noise=trans_noise)
    # --- No Wall Gridworld ---
    # P = gridworld_transition_matrix(int(np.sqrt(n)), int(np.sqrt(n)), stay_prob=(1.0-mixing_coef), stochasticity=noise_coef)
    # --- Four Rooms Gridworld ---
    free = make_four_rooms(int(np.sqrt(n)), int(np.sqrt(n)), wall_thickness=1, door_size=2, border=1)
    P = gridworld_transition_matrix_with_walls(int(np.sqrt(n)), int(np.sqrt(n)), free, stay_prob=(1.0-mixing_coef), remove_wall_states=remove_wall_states)
    return P


def low_rank_approximation(M, r, use_highest=True):
    """
    Compute the rank-r approximation of matrix M using SVD.
    Args:
        M: input matrix (n x n)
        r: target rank
        use_highest: if True, use top-r singular values; else use bottom-r singular values
    Returns:
        M_r: rank-r approximation of M
    """
    U, S, Vt = LA.svd(M)
    S_r = np.zeros((M.shape[0], M.shape[1]))
    if use_highest:
        S_r[:r, :r] = np.diag(S[:r])
    else:
        S_r[-r:, -r:] = np.diag(S[-r:])
    M_r = U @ S_r @ Vt
    return M_r

def stable_rank(M):
    """
    Compute the stable rank of matrix M.
    Args:
        M: input matrix (n x n)
    Returns:
        stable_rank: stable rank of M
    """
    frob_norm = LA.norm(M, 'fro')
    op_norm = LA.norm(M, 2)
    return (frob_norm ** 2) / (op_norm ** 2)


def participation_ratio(M, tol=1e-12):
    """
    Participation Ratio of matrix M.

    Parameters
    ----------
    M : np.ndarray
        SR matrix (or approximation)
    tol : float
        Singular values below this are ignored

    Returns
    -------
    float
        Participation ratio
    """
    s = np.linalg.svd(M, compute_uv=False)
    s = s[s > tol]
    s2 = s ** 2

    numerator = np.sum(s2) ** 2
    denominator = np.sum(s2 ** 2)

    if denominator == 0.0:
        return 0.0

    return numerator / denominator


def spectral_entropy(M, tol=1e-12):
    """
    Entropy-based effective rank of matrix M.

    Parameters
    ----------
    M : np.ndarray
        SR matrix (or approximation)
    tol : float
        Singular values below this are ignored

    Returns
    -------
    float
        Entropy effective rank
    """
    s = np.linalg.svd(M, compute_uv=False)
    s = s[s > tol]
    s2 = s ** 2
    p = s2 / np.sum(s2)

    # Numerical safety: avoid log(0)
    entropy = -np.sum(p * np.log(p + 1e-20))
    # Normalize by log(B) where B is the number of components
    # This keeps the metric in the range [0, 1]
    normalization_factor = np.log(len(s))

    return entropy / normalization_factor if normalization_factor > 0 else 0.0



## Experiments:
#### 1. P is Identity Matrix (mixing=0.0, gamma=0.99)
- When P is Identity, all eigenvalues of M are $1/(1-\gamma)$
- States do not mix, so all eigenvalues are the same.
- Each state is its own cluster.
- Low-rank approximation of M does not work well since all eigenvalues are the same.
- **Reconstruction error is very high especially for low ranks.**

#### 2. P moves away from Identity (e.g., mixing_coef=0.0 -> 0.5, gamma=0.99)
- We start to see a slight spread in the eigenvalues of P and M.
- M eigenvalues are scaled versions of P eigenvalues.
- Some states start to form clusters based on transition probabilities.
- Low-rank approximation of M starts to work better as some eigenvalues are larger.
- **Reconstruction error decreases SIGNIFICANTLY for low ranks compared to the Identity case.**
#### 2. P near Identity and $\gamma$ increasing (mixing_coef:0.05, gamma: 0.0 -> 0.99)
- As $\gamma$ increases, eigenvalues of M become larger and more spread out.
- M changes to reflect longer-term dynamics.
- In fact for M to be able to capture long-term dynamics, $\gamma$ needs to be very high (close to 1).
- However very high $\gamma$ with P near Identity leads to numerical instability. (eigenvalues of M become extremely large)
- **Reconstruction error increases as $\gamma$ increases, and very high $\gamma$ can lead to instability.**
#### 3. P is mixing and $\gamma$ is increasing (mixing_coef=0.5, gamma= 0.0 -> 0.99)
- Eigenvalues of P are more spread out due to mixing.
- M shows long-term dynamics more clearly and at lower $\gamma$ values. (lower $\gamma$ can capture long-term dynamics when P is mixing)
- Reconstruction error remains relatively low across a range of $\gamma$ values.
- **Mixing in P helps maintain low reconstruction error even as $\gamma$ increases.**

#### compare the M matrix visually:
1.  P near Identiry and high gamma (mixing_coef=0.05, gamma=0.99): Reconstruction (rank10) -> ~ 125
2.  P mixing and lower gamma (mixing_coef=0.5, gamma=0.90): Reconstruction (rank10) -> ~15
- However the M matrix looks similar in both cases!

---

## TAKEAWAY:
Instead on increasing gamma to capture long-term dynamics, it is more effective and stable to increase the mixing in the transition matrix P. This also allows us to maintain a low-rank structure in the successor representation M, facilitating efficient approximations and reconstructions.


In [ ]:
# Global state (or wrap in a closure if you prefer)
fig, ax = plt.subplots(5, 5, figsize=(16, 12))
handles = {
    "p_img": None,
    "pk_img": None,
    "l_img": None,
    "lk_img": None,
    "m_img": None,
    "mk_img": None,
    "m_top_img": None,
    "mk_top_img": None,
    "m_bottom_img": None,
    "mk_bottom_img": None,
    "eig_p": None,
    "eig_pk": None,
    "eig_l": None,
    "eig_lk": None,
    "eig_m": None,
    "eig_mk": None,
    "err_top": None,
    "err_k_top": None,
    "err_bottom": None,
    "err_k_bottom": None,
    "colorbars": [None] * 10,
}

def plot_transition_and_sr(N, gamma, mixing_coef, noise_coef, reconstruction_rank, K):
    use_eigendecomposition = False
    
    P = setup(N, mixing_coef=mixing_coef, noise_coef=noise_coef, remove_wall_states=False)
    L = np.eye(P.shape[0]) - gamma * P
    M = LA.pinv(np.eye(P.shape[0]) - gamma * P)
    PK = np.linalg.matrix_power(P, K)  # P after K steps of markov chain
    MK = LA.pinv(np.eye(PK.shape[0]) - gamma * PK)
    LK = np.eye(PK.shape[0]) - gamma * PK

    if use_eigendecomposition:
        eig_P = np.sort(np.real(LA.eig(P)[0]))[::-1]
        eig_L = np.sort(np.real(LA.eig(L)[0]))[::-1]
        eig_M = np.sort(np.real(LA.eig(M)[0]))[::-1]

        eig_PK = np.sort(np.real(LA.eig(PK)[0]))[::-1]
        eig_LK = np.sort(np.real(LA.eig(LK)[0]))[::-1]
        eig_MK = np.sort(np.real(LA.eig(MK)[0]))[::-1]
    else: # use SVD-based spectrum approximation
        eig_P = LA.svd(P, compute_uv=False)**2
        eig_L = LA.svd(L, compute_uv=False)**2
        eig_M = LA.svd(M, compute_uv=False)**2

        eig_PK = LA.svd(PK, compute_uv=False)**2
        eig_LK = LA.svd(LK, compute_uv=False)**2
        eig_MK = LA.svd(MK, compute_uv=False)**2



    # --- Matrices ---
    if handles["p_img"] is None:
        handles["p_img"] = ax[0, 0].imshow(P, cmap="viridis_r")
        ax[0, 0].set_title("Transition Matrix P")
        ax[0, 0].set_xticks([]); ax[0, 0].set_yticks([])
        handles["colorbars"][0] = plt.colorbar(handles["p_img"], ax=ax[0, 0])
    else:
        handles["p_img"].set_data(P)
        handles["p_img"].autoscale()

    if handles["l_img"] is None:
        handles["l_img"] = ax[0, 1].imshow(L, cmap="viridis_r")
        ax[0, 1].set_title("Laplacian Matrix $L(P)$")
        ax[0, 1].set_xticks([]); ax[0, 1].set_yticks([])
        handles["colorbars"][1] = plt.colorbar(handles["l_img"], ax=ax[0, 1])
    else:
        handles["l_img"].set_data(L)
        handles["l_img"].autoscale()

    if handles["m_img"] is None:
        handles["m_img"] = ax[0, 2].imshow(M, cmap="viridis_r")
        ax[0, 2].set_title("Successor Representation $M(P)$")
        ax[0, 2].set_xticks([]); ax[0, 2].set_yticks([])
        handles["colorbars"][2] = plt.colorbar(handles["m_img"], ax=ax[0, 2])
    else:
        handles["m_img"].set_data(M)
        handles["m_img"].autoscale()

    if handles["pk_img"] is None:
        handles["pk_img"] = ax[2, 0].imshow(PK, cmap="viridis_r")
        ax[2, 0].set_title(f"Transition Matrix $P^K$")
        ax[2, 0].set_xticks([]); ax[2, 0].set_yticks([])
        handles["colorbars"][5] = plt.colorbar(handles["pk_img"], ax=ax[2, 0])
    else:
        handles["pk_img"].set_data(PK)
        handles["pk_img"].autoscale()
    
    if handles["lk_img"] is None:
        handles["lk_img"] = ax[2, 1].imshow(LK, cmap="viridis_r")
        ax[2, 1].set_title(f"Laplacian Matrix $L(P^K)$")
        ax[2, 1].set_xticks([]); ax[2, 1].set_yticks([])
        handles["colorbars"][6] = plt.colorbar(handles["lk_img"], ax=ax[2, 1])
    else:
        handles["lk_img"].set_data(LK)
        handles["lk_img"].autoscale()
    
    if handles["mk_img"] is None:
        handles["mk_img"] = ax[2, 2].imshow(MK, cmap="viridis_r")
        ax[2, 2].set_title(f"Successor Representation $M(P^K)$")
        ax[2, 2].set_xticks([]); ax[2, 2].set_yticks([])
        handles["colorbars"][7] = plt.colorbar(handles["mk_img"], ax=ax[2, 2])
    else:
        handles["mk_img"].set_data(MK)
        handles["mk_img"].autoscale()

    # --- Reconstructions ---
    M_top = low_rank_approximation(M, reconstruction_rank, use_highest=True)
    if handles["m_top_img"] is None:
        handles["m_top_img"] = ax[0, 3].imshow(M_top, cmap="viridis_r")
        ax[0, 3].set_title("Top-r Rec. of $M(P)$")
        ax[0, 3].set_xticks([]); ax[0, 3].set_yticks([])
        handles["colorbars"][3] = plt.colorbar(handles["m_top_img"], ax=ax[0, 3])
    else:
        handles["m_top_img"].set_data(M_top)
        handles["m_top_img"].autoscale()

    M_bottom = low_rank_approximation(M, reconstruction_rank, use_highest=False)
    if handles["m_bottom_img"] is None:
        handles["m_bottom_img"] = ax[0, 4].imshow(M_bottom, cmap="viridis_r")
        ax[0, 4].set_title("Bottom-r Rec. $M(P)$")
        ax[0, 4].set_xticks([]); ax[0, 4].set_yticks([])
        handles["colorbars"][4] = plt.colorbar(handles["m_bottom_img"], ax=ax[0, 4])
    else:
        handles["m_bottom_img"].set_data(M_bottom)
        handles["m_bottom_img"].autoscale()

    MK_top = low_rank_approximation(MK, reconstruction_rank, use_highest=True)
    if handles["mk_top_img"] is None:
        handles["mk_top_img"] = ax[2, 3].imshow(MK_top, cmap="viridis_r")
        ax[2, 3].set_title(f"Top-r Rec. of $M(P^K)$")
        ax[2, 3].set_xticks([]); ax[2, 3].set_yticks([])
        handles["colorbars"][8] = plt.colorbar(handles["mk_top_img"], ax=ax[2, 3])
    else:
        handles["mk_top_img"].set_data(MK_top)
        handles["mk_top_img"].autoscale()

    MK_bottom = low_rank_approximation(MK, reconstruction_rank, use_highest=False)
    if handles["mk_bottom_img"] is None:
        handles["mk_bottom_img"] = ax[2, 4].imshow(MK_bottom, cmap="viridis_r")
        ax[2, 4].set_title(f"Bottom-r Rec. of $M(P^K)$")
        ax[2, 4].set_xticks([]); ax[2, 4].set_yticks([])
        handles["colorbars"][9] = plt.colorbar(handles["mk_bottom_img"], ax=ax[2, 4])
    else:
        handles["mk_bottom_img"].set_data(MK_bottom)
        handles["mk_bottom_img"].autoscale()

    # --- Eigen spectra ---
    x_idx = np.arange(len(eig_P))
    if handles["eig_p"] is None:
        (handles["eig_p"],) = ax[1, 0].plot(x_idx, eig_P, marker="o")
        ax[1, 0].set_title("Eigenvalues of $P$")    
        ax[1, 0].grid(True)
    else:
        handles["eig_p"].set_data(x_idx, eig_P)
        ax[1, 0].relim(); ax[1, 0].autoscale_view()

    x_idx = np.arange(len(eig_L))
    if handles["eig_l"] is None:
        (handles["eig_l"],) = ax[1, 1].plot(x_idx, eig_L, marker="o", color="red")
        ax[1, 1].set_title("Eigenvalues of $L(P)$")
        ax[1, 1].grid(True)
    else:
        handles["eig_l"].set_data(x_idx, eig_L)
        ax[1, 1].relim(); ax[1, 1].autoscale_view()

    x_idx = np.arange(len(eig_M))
    if handles["eig_m"] is None:
        (handles["eig_m"],) = ax[1, 2].plot(x_idx, eig_M, marker="o", color="green")
        ax[1, 2].set_title("Eigenvalues of $M(P)$")
        ax[1, 2].grid(True)
    else:
        handles["eig_m"].set_data(x_idx, eig_M)
        ax[1, 2].relim(); ax[1, 2].autoscale_view()

    x_idx = np.arange(len(eig_PK))
    if handles["eig_pk"] is None:
        (handles["eig_pk"],) = ax[3, 0].plot(x_idx, eig_PK, marker="o", color="orange")
        ax[3, 0].set_title(f"Eigenvalues of $P^K$")
        ax[3, 0].grid(True)
    else:
        handles["eig_pk"].set_data(x_idx, eig_PK)
        ax[3, 0].relim(); ax[3, 0].autoscale_view()

    x_idx = np.arange(len(eig_LK))
    if handles["eig_lk"] is None:
        (handles["eig_lk"],) = ax[3, 1].plot(x_idx, eig_LK, marker="o", color="red")
        ax[3, 1].set_title(f"Eigenvalues of $L(P^K)$")
        ax[3, 1].grid(True)
    else:
        handles["eig_lk"].set_data(x_idx, eig_LK)
        ax[3, 1].relim(); ax[3, 1].autoscale_view()
    
    x_idx = np.arange(len(eig_MK))
    if handles["eig_mk"] is None:
        (handles["eig_mk"],) = ax[3, 2].plot(x_idx, eig_MK, marker="o", color="green")
        ax[3, 2].set_title(f"Eigenvalues of $M(P^K)$")
        ax[3, 2].grid(True)
    else:
        handles["eig_mk"].set_data(x_idx, eig_MK)
        ax[3, 2].relim(); ax[3, 2].autoscale_view()

    # --- Error curves ---
    ranks = np.arange(1, N)

    err_top = [LA.norm(M - low_rank_approximation(M, r, True), ord="fro") for r in ranks]
    if handles["err_top"] is None:
        (handles["err_top"],) = ax[1, 3].plot(ranks, err_top, marker="o")
        ax[1, 3].set_title("Error vs Rank \n(Top-r) of $M(P)$")
        ax[1, 3].set_xlabel("Rank r"); ax[1, 3].set_ylabel("Frobenius Error")
        ax[1, 3].grid(True)
    else:
        handles["err_top"].set_data(ranks, err_top)
        ax[1, 3].relim(); ax[1, 3].autoscale_view()

    err_bottom = [LA.norm(M - low_rank_approximation(M, r, False), ord="fro") for r in ranks]
    if handles["err_bottom"] is None:
        (handles["err_bottom"],) = ax[1, 4].plot(ranks, err_bottom, marker="o", color="orange")
        ax[1, 4].set_title("Error vs Rank \n(Bottom-r) of $M(P)$")
        ax[1, 4].set_xlabel("Rank r"); ax[1, 4].set_ylabel("Frobenius Error")
        ax[1, 4].grid(True)
    else:
        handles["err_bottom"].set_data(ranks, err_bottom)
        ax[1, 4].relim(); ax[1, 4].autoscale_view()

    err_k_top = [LA.norm(MK - low_rank_approximation(MK, r, True), ord="fro") for r in ranks]
    if handles["err_k_top"] is None:
        (handles["err_k_top"],) = ax[3, 3].plot(ranks, err_k_top, marker="o")
        ax[3, 3].set_title(f"Error vs Rank \n(Top-r) of $M(P^K)$")
        ax[3, 3].set_xlabel("Rank r"); ax[3, 3].set_ylabel("Frobenius Error")
        ax[3, 3].grid(True)
    else:
        handles["err_k_top"].set_data(ranks, err_k_top)
        ax[3, 3].relim(); ax[3, 3].autoscale_view()

    err_k_bottom = [LA.norm(MK - low_rank_approximation(MK, r, False), ord="fro") for r in ranks]
    if handles["err_k_bottom"] is None:
        (handles["err_k_bottom"],) = ax[3, 4].plot(ranks, err_k_bottom, marker="o", color="orange")
        ax[3, 4].set_title(f"Error vs Rank \n(Bottom-r) of $M(P^K)$")
        ax[3, 4].set_xlabel("Rank r"); ax[3, 4].set_ylabel("Frobenius Error")
        ax[3, 4].grid(True)
    else:
        handles["err_k_bottom"].set_data(ranks, err_k_bottom)
        ax[3, 4].relim(); ax[3, 4].autoscale_view()

    # --- rank comparisons ---
    sr_M = stable_rank(M)
    sr_MK = stable_rank(MK)
    ax[4, 0].clear()
    ax[4, 0].bar(["$M(P)$", "$M(P^K)$"], [sr_M, sr_MK], color=["blue", "green"])
    ax[4, 0].set_title("Stable Rank Comparison")
    ax[4, 0].set_ylabel("Stable Rank")

    pr_M = participation_ratio(M)
    pr_MK = participation_ratio(MK)
    ax[4, 1].clear()
    ax[4, 1].bar(["$M(P)$", "$M(P^K)$"], [pr_M, pr_MK], color=["blue", "green"])
    ax[4, 1].set_title("Participation Ratio Comparison")
    ax[4, 1].set_ylabel("Participation Ratio")

    er_M = spectral_entropy(M)
    er_MK = spectral_entropy(MK)
    ax[4, 2].clear()
    ax[4, 2].bar(["$M(P)$", "$M(P^K)$"], [er_M, er_MK], color=["blue", "green"])
    ax[4, 2].set_title("Spectral Entropy Comparison")
    ax[4, 2].set_ylabel("Spectral Entropy")

    # Hide unused subplots
    ax[4, 3].axis('off')
    ax[4, 4].axis('off')

    fig.tight_layout()
    fig.canvas.draw()

_=interact(
    plot_transition_and_sr,
    N=IntSlider(min=4, max=100, step=1, value=16, description="N"),
    K=IntSlider(min=1, max=10, step=1, value=1, description="K"),
    gamma=FloatSlider(min=0.0, max=0.999, step=0.001, value=0.99, description="Gamma", readout_format=".3f"),
    mixing_coef=FloatSlider(min=0.0, max=1.0, step=0.01, value=0.0, description="P Mixing"),
    noise_coef=FloatSlider(min=0.0, max=1.0, step=0.05, value=0.0, description="P Noise"),
    reconstruction_rank=IntSlider(min=1, max=100, step=1, value=10, description="Recon. Rank"),
)

### Paper Plot1: Change in K

In [ ]:
def beautify_axis_x(
    ax, x_limits, ax_offset_scale=0.05, xticks_int=False, decimals: int = 1
):
    x_range = abs(x_limits[1] - x_limits[0])
    ax.set_xlim(
        x_limits[0] - ax_offset_scale * x_range, x_limits[1] + ax_offset_scale * x_range
    )
    # round to 1 decimal place
    x_limits = np.round(x_limits, decimals)
    xticks = np.linspace(*x_limits, 3)
    if xticks_int:
        xticks = xticks.astype(int)
    ax.set_xticks(xticks)
    ax.spines["bottom"].set_bounds(*x_limits)

def beautify_axis_y(
    ax, y_limits, ax_offset_scale=0.05, decimals: int = 1
):
    y_range = abs(y_limits[1] - y_limits[0])
    ax.set_ylim(
        y_limits[0] - ax_offset_scale * y_range, y_limits[1] + ax_offset_scale * y_range
    )
    # round to 1 decimal place
    y_limits = np.round(y_limits, decimals)
    yticks = np.linspace(*y_limits, 3)
    ax.set_yticks(yticks)
    ax.spines["left"].set_bounds(*y_limits)

def beautify_axis(
    ax, x_limits=None, y_limits=None, ax_offset_scale=0.05, xticks_int=False, decimals: int = 1
):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    if x_limits:
        beautify_axis_x(ax, x_limits=x_limits, ax_offset_scale=ax_offset_scale, xticks_int=xticks_int)
    if y_limits:
        beautify_axis_y(ax, y_limits=y_limits, ax_offset_scale=ax_offset_scale, decimals=decimals)



In [ ]:
N = 100
noise = 0.0
mixing_coef = 0.1
PLOT_SAVE_PATH = Path("../plots/discrete_spectral_analysis")
PLOT_SAVE_PATH.mkdir(parents=True, exist_ok=True)


# ============================================================
# --- Varying K - gamma=0.9
# ============================================================
gamma = 0.9
numk = 9
M_list = []
S_list = []

P = setup(N, mixing_coef=mixing_coef, noise_coef=noise, remove_wall_states=True)
M = LA.inv(np.eye(P.shape[0]) - gamma * P)
M_list.append(M)
S = LA.svd(M, compute_uv=False)
S_list.append(S)
for k in range(2, numk + 1):
    Pk = np.linalg.matrix_power(P, k)
    Mk = LA.inv(np.eye(Pk.shape[0]) - gamma * Pk)
    M_list.append(Mk)
    Sk = LA.svd(Mk, compute_uv=False)
    S_list.append(Sk)


spectral_entropies = [spectral_entropy(M) for M in M_list]
stable_ranks = [stable_rank(M) for M in M_list]
condition_numbers = [LA.cond(M) for M in M_list]

# make a list of colors that go from orange to blue dynamically given the number of K values
colors = plt.get_cmap('cool', numk)
fig, ax = plt.subplots(nrows=1, ncols=4, figsize=(14, 4))
for i, s in enumerate(S_list, 1):
    ax[0].plot(s, marker="o", label=f"K={i}", markersize=3, linewidth=1, color=colors(i))
beautify_axis(ax[0], x_limits=(0, P.shape[0]), y_limits=(1, 10), xticks_int=True, decimals=0)
ax[0].set_title("Singular Values of $M$ [$\gamma$=0.9]", fontsize=16)
ax[0].set_xlabel("Index", fontsize=14)
ax[0].set_ylabel("Singular Value", fontsize=14)
ax[0].tick_params(axis='both', which='major', labelsize=12)
ax[0].legend()

# as scatter + line plots
ax[1].plot(range(1, numk + 1), spectral_entropies, marker="o",  markersize=10,  color="slateblue", linewidth=2)
beautify_axis(ax[1], x_limits=(1, numk), y_limits=(0, 1), xticks_int=True, decimals=1)
ax[1].set_title("Spectral Entropy of $M$ [$\gamma$=0.9]", fontsize=16)
ax[1].set_ylabel("Spectral Entropy", fontsize=14)
ax[1].set_xlabel("K", fontsize=14)
ax[1].tick_params(axis='both', which='major', labelsize=12)

ax[2].plot(range(1, numk + 1), stable_ranks, marker="o",  markersize=10,  color="slateblue", linewidth=2)
beautify_axis(ax[2], x_limits=(1, numk), y_limits=(0, 20), xticks_int=True, decimals=1)
ax[2].set_title("Stable Rank of $M$ [$\gamma$=0.9]", fontsize=16)
ax[2].set_ylabel("Stable Rank", fontsize=14)
ax[2].set_xlabel("K", fontsize=14)
ax[2].tick_params(axis='both', which='major', labelsize=12)

ax[3].plot(range(1, numk + 1), condition_numbers, marker="o",  markersize=10,  color="slateblue", linewidth=2)
beautify_axis(ax[3], x_limits=(1, numk), y_limits=(1, 10), xticks_int=True, decimals=1)
ax[3].set_title("Condition Number of $M$ [$\gamma$=0.9]", fontsize=16)
ax[3].set_ylabel("Condition Number", fontsize=14)
ax[3].set_xlabel("K", fontsize=14)
ax[3].tick_params(axis='both', which='major', labelsize=12)

fig.tight_layout(w_pad=3.0)
fig.savefig(PLOT_SAVE_PATH / "k_sensitivity.pdf")
plt.show()



# ============================================================
# --- Varying gamma - K=1
# ============================================================
k = 1
ngammas = 8
gammas = np.linspace(0.9, 0.999, ngammas)
M_list = []
S_list = []

P = setup(N, mixing_coef=mixing_coef, noise_coef=noise, remove_wall_states=True)
PK = np.linalg.matrix_power(P, k)
for gamma in gammas:
    M = LA.inv(np.eye(PK.shape[0]) - gamma * PK)
    M_list.append(M)
    S = LA.svd(M, compute_uv=False)
    S_list.append(S)


spectral_entropies = [spectral_entropy(M) for M in M_list]
stable_ranks = [stable_rank(M) for M in M_list]
condition_numbers = [LA.cond(M) for M in M_list]
# make a list of colors that go from orange to blue dynamically given the number of K values
colors = plt.get_cmap('cool', ngammas)
fig, ax = plt.subplots(nrows=1, ncols=4, figsize=(14, 4))
for i, s in enumerate(S_list, 0):
    ax[0].plot(s, marker="o", label=f"$\gamma={gammas[i]:.2f}$", markersize=3, linewidth=1, color=colors(i))
beautify_axis(ax[0], x_limits=(0, P.shape[0]), xticks_int=True, decimals=0)
# ax[0].spines['left'].set_bounds(3, 1000)
ax[0].set_yscale("symlog")
ax[0].set_ylim(bottom=2)
ax[0].set_title(f"Singular Values of $M$ [k=1]", fontsize=16)
ax[0].set_xlabel("Index", fontsize=14)
ax[0].set_ylabel("Singular Value", fontsize=14)
ax[0].tick_params(axis='both', which='major', labelsize=12)
ax[0].legend()

# as scatter + line plots
ax[1].plot(gammas, spectral_entropies, marker="o",  markersize=10,  color="slateblue", linewidth=2)
beautify_axis(ax[1], x_limits=(0.9, 1.0), y_limits=(0, 1), xticks_int=False, decimals=1)
ax[1].set_title(f"Spectral Entropy of $M$ [k=1]", fontsize=16)
ax[1].set_ylabel("Spectral Entropy", fontsize=14)
ax[1].set_xlabel("$\gamma$", fontsize=14)
ax[1].tick_params(axis='both', which='major', labelsize=12)

beautify_axis(ax[2], x_limits=(0.9, 1.0), y_limits=(0, 20), xticks_int=False, decimals=1)
ax[2].plot(gammas, stable_ranks, marker="o",  markersize=10,  color="slateblue", linewidth=2)
ax[2].set_title(f"Stable Rank of $M$ [k=1]", fontsize=16)
ax[2].set_ylabel("Stable Rank", fontsize=14)
ax[2].set_xlabel("$\gamma$", fontsize=14)
ax[2].tick_params(axis='both', which='major', labelsize=12)

beautify_axis(ax[3], x_limits=(0.9, 1.0), y_limits=(1, 250), xticks_int=False, decimals=1)
ax[3].plot(gammas, condition_numbers, marker="o",  markersize=10,  color="slateblue", linewidth=2)
ax[3].set_title(f"Condition Number of $M$ [k=1]", fontsize=16)
ax[3].set_ylabel("Condition Number", fontsize=14)
ax[3].set_xlabel("$\gamma$", fontsize=14)
ax[3].tick_params(axis='both', which='major', labelsize=12)

fig.tight_layout(w_pad=3.0)
fig.savefig(PLOT_SAVE_PATH / "gamma_sensitivity.pdf")
plt.show()


# ============================================================
# --- Varying K and gamma together ---
# ============================================================
P = setup(N, mixing_coef=mixing_coef, noise_coef=noise, remove_wall_states=True)
k_values = np.arange(1, 12, 1)
gamma_values = np.linspace(0.9, 0.999, 10)
fig, ax = plt.subplots(nrows=1, ncols=3, figsize=(10, 4))
k_list = []
gamma_list = []
entropy_list = []
stable_rank_list = []
condition_number_list = []
for k in k_values:
    PK = np.linalg.matrix_power(P, k)
    for gamma in gamma_values:
        M = LA.inv(np.eye(PK.shape[0]) - gamma * PK)
        entropy = spectral_entropy(M)
        stable_rank_value = stable_rank(M)
        condition_number = LA.cond(M)
        k_list.append(k)
        gamma_list.append(gamma)
        entropy_list.append(entropy)
        stable_rank_list.append(stable_rank_value)
        condition_number_list.append(condition_number)


entropy_img = np.array(entropy_list).reshape(len(k_values), len(gamma_values)).swapaxes(0, 1)
img1 = ax[0].imshow(entropy_img, aspect='auto', cmap='cool', origin='lower', extent=[min(k_values), max(k_values), min(gamma_values), max(gamma_values)+0.001])
beautify_axis(ax[0], x_limits=(1, len(k_values)), y_limits=(0.9, 1.0), xticks_int=True, decimals=3)
ax[0].spines[["right", "left", "top", "bottom"]].set_visible(False)
ax[0].set_title("Spectral Entropy of $M$", fontsize=16)
ax[0].set_xlabel("K", fontsize=14)
ax[0].set_ylabel("$\gamma$", fontsize=14)
ax[0].tick_params(axis='both', which='major', labelsize=12)
fig.colorbar(img1, ax=ax[0])


stable_rank_img = np.array(stable_rank_list).reshape(len(k_values), len(gamma_values)).swapaxes(0, 1)
img3 = ax[1].imshow(stable_rank_img, aspect='auto', cmap='cool', origin='lower', extent=[min(k_values), max(k_values), min(gamma_values), max(gamma_values)+0.001])
beautify_axis(ax[1], x_limits=(1, len(k_values)), y_limits=(0.9, 1.0), xticks_int=True, decimals=3)
ax[1].spines[["right", "left", "top", "bottom"]].set_visible(False)
ax[1].set_title("Stable Rank of $M(P^K)$", fontsize=16)
ax[1].set_xlabel("K", fontsize=14)
ax[1].set_ylabel("$\gamma$", fontsize=14)
ax[1].tick_params(axis='both', which='major', labelsize=12)
fig.colorbar(img3, ax=ax[1])

condition_number_img = np.array(condition_number_list).reshape(len(k_values), len(gamma_values)).swapaxes(0, 1)
img2 = ax[2].imshow(condition_number_img, aspect='auto', cmap='cool', origin='lower', extent=[min(k_values), max(k_values), min(gamma_values), max(gamma_values)+0.001])
beautify_axis(ax[2], x_limits=(1, len(k_values)), y_limits=(0.9, 1.0), xticks_int=True, decimals=3)
ax[2].spines[["right", "left", "top", "bottom"]].set_visible(False)
ax[2].set_title("Condition Number of $M$", fontsize=16)
ax[2].set_xlabel("K", fontsize=14)
ax[2].set_ylabel("$\gamma$", fontsize=14)
ax[2].tick_params(axis='both', which='major', labelsize=12)
fig.colorbar(img2, ax=ax[2])


fig.tight_layout()
plt.show()


In [ ]:
# SAME AS ABOVE BUT WITH INDIVIDUAL PLTS
N = 100
noise = 0.0
mixing_coef = 0.1
PLOT_SAVE_PATH = Path("../plots/discrete_spectral_analysis")
PLOT_SAVE_PATH.mkdir(parents=True, exist_ok=True)
title_size = 24
label_size = 24
tick_size = 18
legend_size = 16


# ============================================================
# --- Varying K - gamma=0.9
# ============================================================
gamma = 0.9
ks = [1,3,5,10,20]
M_list = []
S_list = []

P = setup(N, mixing_coef=mixing_coef, noise_coef=noise, remove_wall_states=True)
M = LA.inv(np.eye(P.shape[0]) - gamma * P)
M_list.append(M)
S = LA.svd(M, compute_uv=False)
S_list.append(S)
for k in ks[1:]:
    Pk = np.linalg.matrix_power(P, k)
    Mk = LA.inv(np.eye(Pk.shape[0]) - gamma * Pk)
    M_list.append(Mk)
    Sk = LA.svd(Mk, compute_uv=False)
    S_list.append(Sk)


spectral_entropies = [spectral_entropy(M) for M in M_list]
stable_ranks = [stable_rank(M) for M in M_list]
condition_numbers = [LA.cond(M) for M in M_list]

# make a list of colors that go from orange to blue dynamically given the number of K values
fig, ax = plt.subplots(figsize=(5, 4))
colors = plt.get_cmap('cool', len(ks))
for i, s in enumerate(S_list, 1):
    ax.plot(s, marker="o", label=f"K={ks[i-1]}", markersize=3, linewidth=1, color=colors(i-1))
beautify_axis(ax, x_limits=(0, P.shape[0]), y_limits=(1, 10), xticks_int=True, decimals=0)
ax.set_title("Singular Values", fontsize=title_size)
ax.set_xlabel("Index", fontsize=label_size)
ax.set_ylabel("Value", fontsize=label_size)
ax.tick_params(axis='both', which='major', labelsize=tick_size)
ax.legend(fontsize=legend_size)
fig.tight_layout(w_pad=3.0)
fig.savefig(PLOT_SAVE_PATH / "singular_values_vary_k.pdf")
plt.close()

# as scatter + line plots
fig, ax = plt.subplots(figsize=(5, 4))
ax.plot(ks, spectral_entropies, marker="o",  markersize=10,  color="slateblue", linewidth=2)
beautify_axis(ax, x_limits=(1, max(ks)), y_limits=(0, 1), xticks_int=True, decimals=1)
ax.set_title("Spectral Entropy", fontsize=title_size)
ax.set_ylabel("Entropy", fontsize=label_size)
ax.set_xlabel("K", fontsize=label_size)
ax.tick_params(axis='both', which='major', labelsize=tick_size)
fig.tight_layout(w_pad=3.0)
fig.savefig(PLOT_SAVE_PATH / "spectral_entropy_vary_k.pdf")
plt.close()

fig, ax = plt.subplots(figsize=(5, 4))
ax.plot(ks, stable_ranks, marker="o",  markersize=10,  color="slateblue", linewidth=2)
beautify_axis(ax, x_limits=(1, max(ks)), y_limits=(0, 20), xticks_int=True, decimals=1)
ax.set_title("Stable Rank", fontsize=title_size)
ax.set_ylabel("Rank", fontsize=label_size)
ax.set_xlabel("K", fontsize=label_size)
ax.tick_params(axis='both', which='major', labelsize=tick_size)
fig.tight_layout(w_pad=3.0)
fig.savefig(PLOT_SAVE_PATH / "stable_rank_vary_k.pdf")
plt.close()

# fig, ax = plt.subplots(figsize=(5, 4))
# ax.plot(range(1, numk + 1), condition_numbers, marker="o",  markersize=10,  color="slateblue", linewidth=2)
# beautify_axis(ax, x_limits=(1, numk), y_limits=(1, 10), xticks_int=True, decimals=1)
# ax.set_title("Condition Number [$\gamma$=0.9]", fontsize=title_size)
# ax.set_ylabel("Condition Number", fontsize=label_size)
# ax.set_xlabel("K", fontsize=label_size)
# ax.tick_params(axis='both', which='major', labelsize=tick_size)
# fig.tight_layout(w_pad=3.0)
# fig.savefig(PLOT_SAVE_PATH / "condition_number_vary-k.pdf")
# plt.close()


# ============================================================
# --- Varying gamma - K=1
# ============================================================
k = 1
# ngammas = 8
# gammas = np.linspace(0.9, 0.999, ngammas)
gammas = [0.9, 0.95, 0.99, 0.999]
M_list = []
S_list = []

P = setup(N, mixing_coef=mixing_coef, noise_coef=noise, remove_wall_states=True)
PK = np.linalg.matrix_power(P, k)
for gamma in gammas:
    M = LA.inv(np.eye(PK.shape[0]) - gamma * PK)
    M_list.append(M)
    S = LA.svd(M, compute_uv=False)
    S_list.append(S)


spectral_entropies = [spectral_entropy(M) for M in M_list]
stable_ranks = [stable_rank(M) for M in M_list]
condition_numbers = [LA.cond(M) for M in M_list]


fig, ax = plt.subplots(figsize=(5, 4))
colors = plt.get_cmap('cool', len(gammas))
for i, s in enumerate(S_list, 0):
    ax.plot(s, marker="o", label=f"$\gamma$={str(gammas[i])}", markersize=3, linewidth=1, color=colors(i))
beautify_axis(ax, x_limits=(0, P.shape[0]), xticks_int=True, decimals=0)
# ax[0].spines['left'].set_bounds(3, 1000)
ax.set_yscale("symlog")
ax.set_ylim(bottom=2)
ax.set_title(f"Singular Values", fontsize=title_size)
ax.set_xlabel("Index", fontsize=label_size)
ax.set_ylabel("Value", fontsize=label_size)
ax.tick_params(axis='both', which='major', labelsize=tick_size)
ax.legend(fontsize=legend_size)
fig.tight_layout(w_pad=3.0)
fig.savefig(PLOT_SAVE_PATH / "singular_values_vary_gamma.pdf")
plt.close()

# as scatter + line plots
fig, ax = plt.subplots(figsize=(5, 4))
ax.plot(gammas, spectral_entropies, marker="o",  markersize=10,  color="slateblue", linewidth=2)
beautify_axis(ax, x_limits=(0.9, 1.0), y_limits=(0, 1), xticks_int=False, decimals=1)
ax.set_title(f"Spectral Entropy", fontsize=title_size)
ax.set_ylabel("Entropy", fontsize=label_size)
ax.set_xlabel("$\gamma$", fontsize=label_size)
ax.tick_params(axis='both', which='major', labelsize=tick_size)
fig.tight_layout(w_pad=3.0)
fig.savefig(PLOT_SAVE_PATH / "spectral_entropy_vary_gamma.pdf")
plt.close()

fig, ax = plt.subplots(figsize=(5, 4))
beautify_axis(ax, x_limits=(0.9, 1.0), y_limits=(0, 20), xticks_int=False, decimals=1)
ax.plot(gammas, stable_ranks, marker="o",  markersize=10,  color="slateblue", linewidth=2)
ax.set_title(f"Stable Rank", fontsize=title_size)
ax.set_ylabel("Rank", fontsize=label_size)
ax.set_xlabel("$\gamma$", fontsize=label_size)
ax.tick_params(axis='both', which='major', labelsize=tick_size)
fig.tight_layout(w_pad=3.0)
fig.savefig(PLOT_SAVE_PATH / "stable_rank_vary_gamma.pdf")
plt.close()

# fig, ax = plt.subplots(figsize=(5, 4))
# beautify_axis(ax, x_limits=(0.9, 1.0), y_limits=(1, 250), xticks_int=False, decimals=1)
# ax.plot(gammas, condition_numbers, marker="o",  markersize=10,  color="slateblue", linewidth=2)
# ax.set_title(f"Condition Number [k=1]", fontsize=title_size)
# ax.set_ylabel("Condition Number", fontsize=label_size)
# ax.set_xlabel("$\gamma$", fontsize=label_size)
# ax.tick_params(axis='both', which='major', labelsize=tick_size)
# fig.tight_layout(w_pad=3.0)
# fig.savefig(PLOT_SAVE_PATH / "condition_number_vary-gamma.pdf")
# plt.close()

# ============================================================
# --- Varying K and gamma together ---
# ============================================================
P = setup(N, mixing_coef=mixing_coef, noise_coef=noise, remove_wall_states=True)
# k_values = np.arange(1, 12, 1)
# gamma_values = np.linspace(0.9, 0.999, 10)
k_values = [1,3,5,10,20]
gamma_values = [0.9, 0.95, 0.99, 0.999]
k_list = []
gamma_list = []
entropy_list = []
stable_rank_list = []
condition_number_list = []
for k in k_values:
    PK = np.linalg.matrix_power(P, k)
    for gamma in gamma_values:
        M = LA.inv(np.eye(PK.shape[0]) - gamma * PK)
        entropy = spectral_entropy(M)
        stable_rank_value = stable_rank(M)
        condition_number = LA.cond(M)
        k_list.append(k)
        gamma_list.append(gamma)
        entropy_list.append(entropy)
        stable_rank_list.append(stable_rank_value)
        condition_number_list.append(condition_number)

fig, ax = plt.subplots(figsize=(5, 4))
entropy_img = np.array(entropy_list).reshape(len(k_values), len(gamma_values)).swapaxes(0, 1)
img1 = ax.imshow(entropy_img, aspect='auto', cmap='cool', origin='lower')
x_tick_labels = [str(k) for k in k_values]
y_tick_labels = [str(gamma) for gamma in gamma_values]
ax.set_xticks(np.arange(len(x_tick_labels)))
ax.set_yticks(np.arange(len(y_tick_labels)))
ax.set_xticklabels(x_tick_labels)
ax.set_yticklabels(y_tick_labels)
ax.spines[["right", "left", "top", "bottom"]].set_visible(False)
ax.set_title("Spectral Entropy", fontsize=title_size)
ax.set_xlabel("K", fontsize=label_size)
ax.set_ylabel("$\gamma$", fontsize=label_size)
ax.tick_params(axis='both', which='major', labelsize=tick_size)
cbar= fig.colorbar(img1, ax=ax)
cbar.ax.locator_params(nbins=4, tight=True)
fig.tight_layout(w_pad=3.0)
fig.savefig(PLOT_SAVE_PATH / "spectral_entropy_heatmap.pdf")
plt.close()

fig, ax = plt.subplots(figsize=(5, 4))
stable_rank_img = np.array(stable_rank_list).reshape(len(k_values), len(gamma_values)).swapaxes(0, 1)
img3 = ax.imshow(stable_rank_img, aspect='auto', cmap='cool', origin='lower')
# beautify_axis(ax, x_limits=(1, max(k_values)), y_limits=(0.9, 1.0), xticks_int=True, decimals=3)
x_tick_labels = [str(k) for k in k_values]
y_tick_labels = [str(gamma) for gamma in gamma_values]
ax.set_xticks(np.arange(len(x_tick_labels)))
ax.set_yticks(np.arange(len(y_tick_labels)))
ax.set_xticklabels(x_tick_labels)
ax.set_yticklabels(y_tick_labels)
ax.spines[["right", "left", "top", "bottom"]].set_visible(False)
ax.set_title("Stable Rank", fontsize=title_size)
ax.set_xlabel("K", fontsize=label_size)
ax.set_ylabel("$\gamma$", fontsize=label_size)
ax.tick_params(axis='both', which='major', labelsize=tick_size)
cbar = fig.colorbar(img3, ax=ax)
cbar.ax.locator_params(nbins=4, tight=True)
fig.tight_layout(w_pad=3.0)
fig.savefig(PLOT_SAVE_PATH / "stable_rank_heatmap.pdf")
plt.close()

# fig, ax = plt.subplots(figsize=(5, 4))
# condition_number_img = np.array(condition_number_list).reshape(len(k_values), len(gamma_values)).swapaxes(0, 1)
# img2 = ax.imshow(condition_number_img, aspect='auto', cmap='cool', origin='lower', extent=[min(k_values), max(k_values), min(gamma_values), max(gamma_values)+0.001])
# beautify_axis(ax, x_limits=(1, len(k_values)), y_limits=(0.9, 1.0), xticks_int=True, decimals=3)
# ax.spines[["right", "left", "top", "bottom"]].set_visible(False)
# ax.set_title("Condition Number", fontsize=title_size)
# ax.set_xlabel("K", fontsize=label_size)
# ax.set_ylabel("$\gamma$", fontsize=label_size)
# ax.tick_params(axis='both', which='major', labelsize=tick_size)
# fig.colorbar(img2, ax=ax)
# fig.tight_layout(w_pad=3.0)
# fig.savefig(PLOT_SAVE_PATH / "condition_number_heatmap.pdf")
# plt.close()

In [ ]:
gamma = 0.999
k = 3
P = setup(N, mixing_coef=mixing_coef, noise_coef=noise, remove_wall_states=True)
Pk = np.linalg.matrix_power(P, k)
M = LA.inv(np.eye(Pk.shape[0]) - gamma * Pk)

M.sum(axis=1)

### The effect of discount factor $\gamma$ and diagonality of the transition matri (controlled by `trans_decay`) on the eigen-spectrum of the successor representation matrix M. 
We also analyze how well low-rank approximations of M perform in terms of Frobenius norm error.

In [ ]:
N = 16 # number of states (should be a perfect square for gridworld)
gammas = (1.0-np.logspace(-2, -0.05, 10))
mixings = np.logspace(-2, -0.05, 10)
ranks = np.arange(1, N+1, 4)

#plot a heatmap of errors for different gammas and decays
error_matrix = np.zeros((len(ranks),len(gammas), len(mixings)))
for k, rank in enumerate(ranks):
    for i, gamma in enumerate(gammas):
        for j, mixing in enumerate(mixings):
            P = setup(N, mixing_coef=mixing, noise_coef=0.0)
            M = LA.inv(np.eye(P.shape[0]) - gamma * P)
            M_r = low_rank_approximation(M, rank)
            error = LA.norm(M - M_r, 'fro') / LA.norm(M, 'fro')
            error_matrix[k, i, j] = error  

# plot heatmap
fig_, ax_ = plt.subplots(nrows=1, ncols=len(ranks), figsize=(20, 4))

for k, rank in enumerate(ranks):
    cax = ax_[k].imshow(error_matrix[k], 
                    extent=[mixings[0], mixings[-1], gammas[0], gammas[-1]],
                    aspect='auto', cmap='viridis')
    ax_[k].set_xlabel('Transition Mixing Coef')
    ax_[k].set_ylabel('Discount Factor γ')
    ax_[k].set_title(f'Approximation Error (Rank={rank})')
    fig_.colorbar(cax, ax=ax_[k], label='Relative Frobenius Norm Error')


### OBSERVATION:
- Low discount factor, $\gamma$, results in lower row-rank reconstruction error for Successor Representation M.
- Low mixing coefficient for transition matrix (more diagonal), results in higher row-rank reconstruction error for Successor Representation M.
### WHY?
---
1. **Eigenvalues of M and P are related**:
$$ M = (I - \gamma P)^{-1} = U (I - \gamma \Lambda)^{-1} U^T $$
where $P = U \Lambda U^T$ is the eigendecomposition of P. So if $\lambda_i$ is an eigenvalue of P, then the corresponding eigenvalue of M is:
$$ \mu_i = \frac{1}{1 - \gamma \lambda_i} $$

Where:
- $\lambda_i$ are the eigenvalues of the dynamics T (ordered $1=\lambda_1 \ge \lambda_2 \dots \ge -1$).
- $\mu_i$ are the eigenvalues of the SR matrix M.
---
2. **The Effect of the Discount Factor ($\gamma$)**:

The discount factor acts as a spectral filter or a non-linear amplifier.

When $\gamma \to 0$ (Myopic):
- The eigenvalues $\mu_i \approx \frac{1}{1} = 1$. The spectrum is flat. The matrix $M$ resembles the Identity matrix.
- *Result*: The matrix is Full Rank. You cannot approximate this well with a low-rank matrix because every state is independent; knowing one state tells you nothing about the future.

When $\gamma \to 1$ (Far-sighted):
- The denominator $(1 - \gamma \lambda_i)$ becomes very sensitive to how close $\lambda_i$ is to 1.
- The largest eigenvalue of $T$ is always $\lambda_1 = 1$. As $\gamma \to 1$, the SR eigenvalue $\mu_1 = \frac{1}{1-\gamma}$ explodes.
- Crucially, for structured environments, there is usually a "spectral gap"—a few eigenvalues are close to 1 (representing slow modes or clusters), and the rest decay quickly.
- High $\gamma$ amplifies this gap. The top few $\mu_i$ become massive, while the noise (small $\lambda_i$) stays small ($\mu_i \approx 1$).
- *Result*: The matrix becomes numerically Low Rank. Most of the "energy" of the matrix is concentrated in the top few components.
---
3. **The Effect of "Diagonality" (Structure of T)**:

It is helpful to contrast Diffusive/Structured dynamics vs. Identity/Random dynamics.

*Case A: The "Diagonal-like" T (Identity or Random Walk)*:

If T≈I (the agent stays where it is) or T is very sparse/random without structure:
- The eigenvalues λi​ are all clustered near 1 (if T≈I) or 0 (if random).
- *Effect on SR*: The SR eigenvalues μi​ will be roughly equal.
- *Low Rank Feasibility*: Poor. The spectrum is flat. You need almost all dimensions to represent M.

*Case B: Structured/Diffusive T (Real World)*:

In realistic dynamics (robotics, navigation), transitions are not diagonal. They are smooth and local. State i transitions to neighbors.
- This creates a Transition Matrix that acts like a diffusion operator (a blurred band matrix).
- *Effect on SR*: This structure causes the eigenvalues λi​ of T to decay. High γ turns this decay into a sharp cutoff in M.
- *Low Rank Feasibility*: Excellent. The "effective rank" is much lower than the number of states.